In [2]:
import os
import numpy as np
import cv2
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Using histogram features for compatibility (no deep learning frameworks needed)

In [4]:
healthy_classes = [
    "Apple", "Avocado", "Banana", "Beef", "Beef-Fillet", "Beef-Goulash", "Beef-Patty",
    "Blueberry-Yogurt", "Grapes", "Grilled-Chicken-Salad", "Grilled-Pizza", "Ground-Beef",
    "Honey", "Jackfruit", "Kiwi", "Lemon", "Lime", "Lychees", "Mango", "Milk", "Mint",
    "Mozzarella", "Orange", "Pancake", "Papaya", "Parsley", "Passion-Fruit", "Peach",
    "Pear", "Pineapple", "Plum", "Raspberries", "Sandwich", "Strawberries", "Tangerine",
    "Watermelon", "Yogurt", "American-Cheese"
]

unhealthy_classes = [
    "BBQ-Chicken-Pizza", "BBQ-Pizza", "Beef-Pizza", "Beef-Ribs", "Beef-Steak", "Biscuit",
    "Cheese-Pizza", "Chicken-Pizza", "Cupcake", "Donut", "Double-Cheeseburger", "Egg-Roll",
    "Hot-Chocoloate", "Hot-Dog", "Mayonnaise", "Muffin", "Mustard", "Pie", "Sausage-Pizza",
    "Seafood-Pizza", "Vanilla-Yogurt", "Vegetable-Pizza", "Vegetarian-Pizza", "Waffles",
    "Zinger-Burger", "Chocolate-Yogurt", "Strawberry-Jam"
]

In [5]:
train_dir = r"c:\Users\stangutur\DS-4002-Group4Project3\DATA\Nutrition Food Proj_Cleaned\Nutrition Food Proj -B0-.folder (1)\train"

X = []
y = []

IMG_SIZE = 224  # Image size for consistent input

for class_folder in os.listdir(train_dir):
    class_path = os.path.join(train_dir, class_folder)

    # skip non-folders
    if not os.path.isdir(class_path):
        continue

    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)

        # Assign label
        if class_folder in healthy_classes:
            label = 1
        elif class_folder in unhealthy_classes:
            label = 0
        else:
            continue  # skip unknown classes

        # Load and preprocess image
        img = cv2.imread(img_path)
        if img is None:
            continue  # skip unreadable images

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

        X.append(img)
        y.append(label)

X = np.array(X)
y = np.array(y)

print("Loaded images:", X.shape)
print("Loaded labels:", y.shape)

Loaded images: (653, 224, 224, 3)
Loaded labels: (653,)


In [6]:
# ---------------------------------------------------------
# 4. FEATURE EXTRACTION (HISTOGRAM OF COLORS)
# ---------------------------------------------------------

def extract_histogram_features(images):
    """Extract color histogram features from images."""
    features = []
    for img in images:
        # Split into BGR channels
        hist_b = cv2.calcHist([img], [0], None, [32], [0, 256])
        hist_g = cv2.calcHist([img], [1], None, [32], [0, 256])
        hist_r = cv2.calcHist([img], [2], None, [32], [0, 256])
        
        # Normalize histograms
        hist_b = cv2.normalize(hist_b, hist_b).flatten()
        hist_g = cv2.normalize(hist_g, hist_g).flatten()
        hist_r = cv2.normalize(hist_r, hist_r).flatten()
        
        # Combine all histograms
        feature_vec = np.concatenate([hist_b, hist_g, hist_r])
        features.append(feature_vec)
    
    return np.array(features)

In [9]:
# ---------------------------------------------------------
# 3. TRAIN/TEST SPLIT
# ---------------------------------------------------------

print("Splitting data into train and test sets...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

Splitting data into train and test sets...
Training set size: 522
Test set size: 131


In [11]:
# ---------------------------------------------------------
# 5. EXTRACT HISTOGRAM FEATURES
# ---------------------------------------------------------

print("Extracting histogram features from training set...")
X_train_features = extract_histogram_features(X_train)
print(f"Training features shape: {X_train_features.shape}")

print("Extracting histogram features from test set...")
X_test_features = extract_histogram_features(X_test)
print(f"Test features shape: {X_test_features.shape}")

# ---------------------------------------------------------
# 6. TRAIN LOGISTIC REGRESSION ON FEATURES
# ---------------------------------------------------------

print("\nTraining Logistic Regression model...")
clf = LogisticRegression(max_iter=2000)
clf.fit(X_train_features, y_train)
print("Model training completed!")

# ---------------------------------------------------------
# 7. EVALUATE MODEL
# ---------------------------------------------------------

y_pred = clf.predict(X_test_features)

print("\n=== MODEL EVALUATION RESULTS ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Extracting histogram features from training set...
Training features shape: (522, 96)
Extracting histogram features from test set...
Test features shape: (131, 96)

Training Logistic Regression model...
Model training completed!

=== MODEL EVALUATION RESULTS ===
Accuracy: 0.7022900763358778

Confusion Matrix:
 [[29 25]
 [14 63]]

Classification Report:
               precision    recall  f1-score   support

           0       0.67      0.54      0.60        54
           1       0.72      0.82      0.76        77

    accuracy                           0.70       131
   macro avg       0.70      0.68      0.68       131
weighted avg       0.70      0.70      0.70       131

